In [1]:
%pip install yfinance

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import csv
import os
import yfinance as yf
from datetime import datetime

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
df = pd.read_csv("constituents.csv")

print("Dataset loaded successfully!")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

df.head()

Dataset loaded successfully!
Number of rows: 503
Number of columns: 8


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


In [4]:
print("Column names:")

print(df.columns.tolist())

Column names:
['Symbol', 'Security', 'GICS Sector', 'GICS Sub-Industry', 'Headquarters Location', 'Date added', 'CIK', 'Founded']


In [5]:
ticker_to_company = dict(
    zip(df["Symbol"], df["Security"])
)

print("Stock dictionary created successfully!")

print(ticker_to_company.get("AAPL"))
print(ticker_to_company.get("MSFT"))

Stock dictionary created successfully!
Apple Inc.
Microsoft


In [6]:
def get_current_price(ticker):

    try:
        # Adjust ticker format for Yahoo Finance
        yahoo_symbol = ticker.replace(".", "-")

        stock = yf.Ticker(yahoo_symbol)

        history = stock.history(period="5d")

        if not history.empty:
            price = history["Close"].dropna().iloc[-1]

            return float(price)

        else:
            print("Online price not available.")

    except Exception as e:
        print("Error retrieving stock price:", e)

    # Manual price entry if online retrieval fails
    return float(input("Enter current price manually: "))

In [7]:
portfolio = {}

print("Portfolio dictionary created successfully!")

Portfolio dictionary created successfully!


In [8]:
number_of_stocks = int(
    input("How many stocks do you want to enter? ")
)

for i in range(number_of_stocks):

    print("\nEnter details for Stock", i + 1)

    ticker = input("Enter stock ticker: ").strip().upper()

    if ticker not in ticker_to_company:
        print("Invalid stock ticker! Try again.")
        continue

    buy_price = float(input("Enter buy price: "))

    quantity = float(input("Enter quantity: "))

    if buy_price <= 0 or quantity <= 0:
        print("Price and quantity must be positive.")
        continue

    current_price = get_current_price(ticker)

    if current_price <= 0:
        print("Invalid current price. Skipping stock.")
        continue

    portfolio[ticker] = {
        "Company": ticker_to_company[ticker],
        "Buy_Price": buy_price,
        "Quantity": quantity,
        "Current_Price": current_price
    }

print("\nAll stock details collected successfully!")

print(portfolio)

How many stocks do you want to enter?  2



Enter details for Stock 1


Enter stock ticker:  AAPL
Enter buy price:  150
Enter quantity:  5



Enter details for Stock 2


Enter stock ticker:  MSFT
Enter buy price:  300
Enter quantity:  2



All stock details collected successfully!
{'AAPL': {'Company': 'Apple Inc.', 'Buy_Price': 150.0, 'Quantity': 5.0, 'Current_Price': 338.32000732421875}, 'MSFT': {'Company': 'Microsoft', 'Buy_Price': 300.0, 'Quantity': 2.0, 'Current_Price': 507.1199951171875}}


In [9]:
records = []

for ticker, details in portfolio.items():

    buy_price = details["Buy_Price"]
    quantity = details["Quantity"]
    current_price = details["Current_Price"]

    # Calculate investment
    invested_amount = buy_price * quantity

    # Calculate current market value
    market_value = current_price * quantity

    # Calculate profit or loss
    profit_loss = market_value - invested_amount

    # Calculate percentage
    profit_percentage = (
        profit_loss / invested_amount
    ) * 100

    record = {
        "Ticker": ticker,
        "Company": details["Company"],
        "Buy_Price": buy_price,
        "Quantity": quantity,
        "Current_Price": current_price,
        "Invested_Amount": invested_amount,
        "Market_Value": market_value,
        "Profit_Loss": profit_loss,
        "Profit_Percentage": profit_percentage
    }

    records.append(record)

print("Portfolio calculations completed!")

Portfolio calculations completed!


In [10]:
portfolio_df = pd.DataFrame(records)

portfolio_df = portfolio_df.round(2)

display(portfolio_df)

,Ticker,Company,Buy_Price,Quantity,Current_Price,Invested_Amount,Market_Value,Profit_Loss,Profit_Percentage
0,AAPL,Apple Inc.,150.0,5.0,338.32,750.0,1691.60,941.60,125.55
1,MSFT,Microsoft,300.0,2.0,507.12,600.0,1014.24,414.24,69.04


In [11]:
total_investment = portfolio_df["Invested_Amount"].sum()

total_market_value = portfolio_df["Market_Value"].sum()

total_profit_loss = total_market_value - total_investment

total_profit_percentage = (
    total_profit_loss / total_investment
) * 100 if total_investment else 0

print("========== PORTFOLIO SUMMARY ==========")

print("Total Investment:", round(total_investment, 2))

print("Total Market Value:", round(total_market_value, 2))

print("Total Profit/Loss:", round(total_profit_loss, 2))

print("Total Profit/Loss Percentage:",
      round(total_profit_percentage, 2), "%")

========== PORTFOLIO SUMMARY ==========
Total Investment: 1350.0
Total Market Value: 2705.84
Total Profit/Loss: 1355.84
Total Profit/Loss Percentage: 100.43 %


In [12]:
filename = "portfolio_summary.csv"

# Add date and time to each record
for record in records:
    record["Date"] = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )

fields = [
    "Date",
    "Ticker",
    "Company",
    "Buy_Price",
    "Quantity",
    "Current_Price",
    "Invested_Amount",
    "Market_Value",
    "Profit_Loss",
    "Profit_Percentage"
]

# Check whether the file already exists
file_exists = os.path.isfile(filename)

# Save records into CSV
with open(filename, "a", newline="") as file:

    writer = csv.DictWriter(file, fieldnames=fields)

    if not file_exists or os.path.getsize(filename) == 0:
        writer.writeheader()

    writer.writerows(records)

print("Portfolio data saved successfully!")
print("File name:", filename)

Portfolio data saved successfully!
File name: portfolio_summary.csv


In [13]:
saved_data = pd.read_csv("portfolio_summary.csv")

display(saved_data)

,Date,Ticker,Company,Buy_Price,Quantity,Current_Price,Invested_Amount,Market_Value,Profit_Loss,Profit_Percentage
0,2026-11-03 17:17:36,AAPL,Apple Inc.,150.0,5.0,338.320007,750.0,1691.600037,941.600037,125.546672
1,2026-11-03 17:17:36,MSFT,Microsoft,300.0,2.0,507.119995,600.0,1014.239990,414.239990,69.039998
